# Embeding flow

                  SAME EMBEDDING MODEL
                         │
        ┌────────────────┼────────────────┐
        ▼                ▼                ▼
   Structure         Parent-child      Semantic
   chunks             chunks            chunks
        │                │                │
        ▼                ▼                ▼
    Database A       Database B       Database C

In [2]:
from docling.document_converter import DocumentConverter
from docling_core.transforms.chunker import HierarchicalChunker
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from qdrant_client import QdrantClient
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
import os

C:\Users\prati\AppData\Local\Temp\ipykernel_8976\2309241963.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


In [2]:
def parse_pdf(pdf_path: str):
    """Parse PDF using Docling."""
    converter = DocumentConverter()
    result = converter.convert(pdf_path)
    return result.document

In [3]:
document = parse_pdf("D:\\AiEnergyAssitanceForBuilding\\PdfResourc\\Used\\ECBC_HVAC_TIP_SHEET_2011.pdf")

[INFO] 2026-09-29 10:45:49,880 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-09-29 10:45:49,896 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\prati\AppData\Local\Programs\Python\Python313\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-09-29 10:45:49,898 [RapidOCR] main.py:65: Using C:\Users\prati\AppData\Local\Programs\Python\Python313\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-09-29 10:45:49,997 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-09-29 10:45:50,001 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\prati\AppData\Local\Programs\Python\Python313\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-09-29 10:45:50,002 [RapidOCR] main.py:65: Using C:\Users\prati\AppData\Local\Programs\Python\Python313\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-09-29 10:45:50,052 [RapidOCR] base

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

In [4]:
chunker = HierarchicalChunker()
chunks = list(chunker.chunk(document))

In [5]:
langchain_chunks = []

for chunk in chunks:
    langchain_chunks.append(
        Document(
            page_content=chunk.text,
            metadata={
                "chunk_type": "hierarchical"
            }
        )
    )

In [6]:

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={
        "device": "cuda"
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)


vector_store = QdrantVectorStore.from_documents(
    documents=langchain_chunks,
    embedding=embeddings,
    path="D:\\M.tech\\Research Area\\RAG pipeline\\DB\\ParentChildChunk",
    collection_name="contextual_token"
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [3]:
client = QdrantClient(path="D:\\M.tech\\Research Area\\RAG pipeline\\DB\\ParentChildChunk")

In [4]:
# Get a list of all collections in that folder

collections_response = client.get_collections()

print("Your available collection names are:")
for collection in collections_response.collections:
    print(f"- {collection.name}")

Your available collection names are:
- contextual_token


In [5]:
# 2. Define the exact same embedding model you used to create the DB
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

# 3. Use the collection name found in your folder structure
vector_store = QdrantVectorStore(
    client=client,
    collection_name="contextual_token",  # <-- This matches your folder name!
    embedding=embeddings,
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [8]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [10]:
docs = retriever.invoke("What is the maximum U-factor for windows?")

In [14]:
docs[1].page_content

'10,000, Table 2: Power Consumption rating for Packaged Air Conditioners-Under Test Conditions.Cooling Capacity.Tons of refrigeration = 3. 10,000, Table 2: Power Consumption rating for Packaged Air Conditioners-Under Test Conditions.Maximum Power Consumption in Watts.Water Cooled = 3,750. 10,000, Table 2: Power Consumption rating for Packaged Air Conditioners-Under Test Conditions.Maximum Power Consumption in Watts.Air Cooled = 4,750. 17,500, Table 2: Power Consumption rating for Packaged Air Conditioners-Under Test Conditions.Cooling Capacity.Tons of refrigeration = 5. 17,500, Table 2: Power Consumption rating for Packaged Air Conditioners-Under Test Conditions.Maximum Power Consumption in Watts.Water Cooled = 6,000. 17,500, Table 2: Power Consumption rating for Packaged Air Conditioners-Under Test Conditions.Maximum Power Consumption in Watts.Air Cooled = 7,000. 26,250, Table 2: Power Consumption rating for Packaged Air Conditioners-Under Test Conditions.Cooling Capacity.Tons of refr